## Datos
Se utilizan datos linealmente separables y con un comportamiento complejo

In [1]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

def GeneraDatos(n, atrib, tipo):
  if tipo==1:
    data, labels = make_classification(n_samples=n, n_features=atrib, n_classes=2, n_clusters_per_class=1,
                           n_redundant=0,class_sep=2.0)
  else:
    data, labels = make_circles(n_samples = n, factor = 0.5, noise = 0.1)
  trainData,testData,trainClass,testClass= train_test_split(data, labels, test_size=0.33, random_state=42)
  plt.scatter(trainData[trainClass == 0, 0], trainData[trainClass == 0, 1], c = "blue")
  plt.scatter(testData[testClass == 0, 0], testData[testClass == 0, 1], c = "skyblue")
  plt.scatter(trainData[trainClass == 1, 0], trainData[trainClass == 1, 1], c = "red")
  plt.scatter(testData[testClass == 1, 0], testData[testClass == 1, 1], c = "salmon")
  plt.axis("equal")
  plt.show()
  return trainData,testData,trainClass,testClass

trainData,testData,trainClass,testClass=GeneraDatos(500,2,2)

## Red neuronal

In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

################## Clasificador ######################
modelo = MLPClassifier(hidden_layer_sizes=(3,3), activation="relu", max_iter=500, learning_rate_init=0.01)

#hidden_layer_sizes=(3, 3),
#activation="logistic" ‘identity’, ‘logistic’, ‘tanh’, ‘relu’}, default=’relu’ (Sigmoide=logistic)
#max_iter=500 #Epocas
#learning_rate_initfloat, default=0.001 learning rate

################## Modelo ###########################
modelo.fit(trainData, trainClass)

##### Clasificar train y test #####################
hipotesisTest=modelo.predict(testData)
hipotesisTrain=modelo.predict(trainData)


#Reporte de clasificación
print("-Conjunto de entrenamiento-\n",classification_report(trainClass,hipotesisTrain))
print("-Conjunto de evaluacion-\n",classification_report(testClass,hipotesisTest))


## Curvas ROC 
Para el cálculo de cirvas ROC, se analiza la probabilidad de predicción (función _.predict_proba()_), la cual retorna la probabilidad de pertencia a las dos clases e orden alfabético, en este caso, primero es la clase 0 y después clase 1. 

Obteniendo de manera automática la curva

In [33]:
from sklearn.metrics import  roc_auc_score
from sklearn.metrics import RocCurveDisplay


probabilidadTest = modelo.predict_proba(testData) #Retorna la probabilidad de que la instancia pertenezca a una clase
print(roc_auc_score(testClass, probabilidadTest[:,1]))

print(testClass[30],probabilidadTest[30])

probTrain = modelo.predict_proba(trainData)
print(roc_auc_score(trainClass, probTrain[:,1])) #0 para la primera clase, 1 para la segunda

#Dos curvas ROC en una figura
fig=plt.figure(figsize=[10,15])
ax1=fig.add_subplot(1,2,1)
ax2=fig.add_subplot(1,2,2)
RocCurveDisplay.from_predictions(trainClass, probTrain[:,1],label='Train', pos_label=1,ax=ax1)
RocCurveDisplay.from_predictions(testClass, probabilidadTest[:,1],label='Test', pos_label=1,ax=ax2)
fig.show()




## Búsqueda de parámetros

In [ ]:
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings("ignore")

# Definir el clasificador
Algoritmo=MLPClassifier()

# Definir los diferentes parámetros a analizar
parametros = {
    'hidden_layer_sizes': [(1),(1,1),(2,2)],
    'activation': ['logistic','identity','logistic', 'tanh', 'relu'],
    'max_iter': [100,200,500],
    'learning_rate_init': [0.001,0.01,0.1]
}

# Realizar la búsqueda en cuadrícula
#(clasificador, parámetros a analizar, validación cruzada (se puede omitir), métrica de interés (se puede omitir))
Busqueda = GridSearchCV(Algoritmo, parametros,scoring='f1_weighted')
Busqueda.fit(trainData, trainClass)

# Obtener los mejores parámetros encontrados
print(Busqueda.best_params_)